# Duke Breast Cancer MRI — 3D Bounding Box Detection - Resnet18
**Multimodal: 3D MRI volume + Clinical features + Radiomic imaging features**

In [1]:
import sys
!{sys.executable} -m pip install torch torchvision opencv-python matplotlib scikit-learn pillow pydicom pynrrd --quiet


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import json, math, warnings
from pathlib import Path
from collections import defaultdict

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import pydicom
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision.models as tv_models

warnings.filterwarnings('ignore')
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

PyTorch  : 2.10.0+cu128
CUDA     : True
GPU      : NVIDIA GeForce RTX 3090
VRAM     : 25.4 GB


## 1. Ground truth check 

In [3]:
def _read_instance_number(path):
    try:
        dcm = pydicom.dcmread(str(path), stop_before_pixels=True)
        return int(getattr(dcm, 'InstanceNumber', 999999))
    except Exception:
        return 999999


def quick_gt_check(export_dir='exported_patients', max_patients=None):
    export_dir = Path(export_dir)
    for split in ('train', 'test'):
        split_dir = export_dir / split
        if not split_dir.exists():
            continue
        patient_dirs = sorted([d for d in split_dir.iterdir() if d.is_dir()])
        if max_patients:
            patient_dirs = patient_dirs[:max_patients]
        sort_bugs = z_bugs = z_ok = 0
        for pd_ in patient_dirs:
            data_file = pd_ / 'patient_data.json'
            mri_dir   = pd_ / 'MRI_DICOM_sample'
            if not data_file.exists() or not mri_dir.exists():
                continue
            with open(data_file) as f:
                meta = json.load(f)
            anns = meta.get('annotations', [])
            if not anns:
                continue
            bbox = anns[0].get('bounding_box_3d', {})
            z_min_gt = bbox.get('start_slice')
            z_max_gt = bbox.get('end_slice')
            if z_min_gt is None:
                continue
            dcm_paths = list(mri_dir.glob('**/*.dcm'))
            if not dcm_paths:
                continue
            instances    = [(_read_instance_number(p), p) for p in dcm_paths]
            inst_sorted  = [p for _, p in sorted(instances, key=lambda x: x[0])]
            alpha_sorted = sorted(dcm_paths)
            if inst_sorted != alpha_sorted:
                sort_bugs += 1
            valid_inst = [i for i, _ in instances if i >= 0]
            if not valid_inst:
                continue
            inst_min, inst_max = min(valid_inst), max(valid_inst)
            overlap  = max(0, min(inst_max, z_max_gt) - max(inst_min, z_min_gt) + 1)
            gt_span  = max(z_max_gt - z_min_gt + 1, 1)
            if 100.0 * overlap / gt_span < 50.0:
                z_bugs += 1
            else:
                z_ok += 1
        print(f'[{split.upper()}] {len(patient_dirs)} patients — '
              f'sort_mismatch={sort_bugs}  z_bug={z_bugs}  z_ok={z_ok}')


quick_gt_check(export_dir='exported_patients')

[TRAIN] 730 patients — sort_mismatch=727  z_bug=0  z_ok=727
[TEST] 178 patients — sort_mismatch=175  z_bug=0  z_ok=175


## 2. Dataset
flips, rotation, z-remapping, lateral extraction 

In [4]:
class Duke3DBBoxDataset(Dataset):
    """
    v6 addition vs v5: breast laterality from DICOM Laterality /
    ImageLaterality tag, encoded as clinical feature 15.
    clinical_dim: 14 → 15  (+1 laterality).
    All other fixes and augmentations unchanged.
    """

    CLINICAL_KEYS = [
        "Age at last contact in EMR f/u(days)(from the date of diagnosis) ,last time patient known to be alive, unless age of death is reported(in such case the age of death",
        "Menopause (at diagnosis)",
        "Days to MRI (From the Date of Diagnosis)",
        "Field Strength (Tesla)",
        "TR (Repetition Time)",
        "TE (Echo Time)",
        "Slice Thickness ",
        "Staging(Nodes)#(Nx replaced by -1)[N]",
        "Staging(Metastasis)#(Mx -replaced by -1)[M]",
        "Staging(Tumor Size)# [T]",
    ]

    IMAGING_KEYS = [
        "TumorMajorAxisLength_mm",
        "Volume_cu_mm_Tumor",
        "Energy_Tumor",
        "Contrast_Tumor",
        "Homogeneity1_Tumor",
        "Max_Enhancement_from_char_curv",
        "Time_to_Peak_from_char_curv",
        "Uptake_rate_from_char_curv",
        "Washout_rate_from_char_curv",
        "breastDensity_T1",
        "breastDensity_PostCon",
        "Peak_SER_tumor",
        "Median_Elongation_Tumor",
    ]

    # Laterality encoding: standard DICOM values → float
    _LAT_MAP = {'L': 0.0, 'LEFT': 0.0, 'R': 1.0, 'RIGHT': 1.0}

    def __init__(
        self, patients_dir, depth=64, image_size=192, augment=False,
        rot_max_deg=10.0,
        clinical_median=None, clinical_mean=None, clinical_std=None,
        imaging_median=None, imaging_mean=None, imaging_std=None,
    ):
        self.patients_dir = Path(patients_dir)
        self.depth        = depth
        self.image_size   = image_size
        self.augment      = augment
        self.rot_max_deg  = rot_max_deg
        self.samples      = []
        # v6: +1 for laterality on top of the +4 (grade, ER, PR, HER2)
        self.clinical_dim = len(self.CLINICAL_KEYS) + 5
        self.imaging_dim  = len(self.IMAGING_KEYS)
        self.clinical_median = clinical_median
        self.clinical_mean   = clinical_mean
        self.clinical_std    = clinical_std
        self.imaging_median  = imaging_median
        self.imaging_mean    = imaging_mean
        self.imaging_std     = imaging_std
        self._load_dataset()

    @staticmethod
    def _read_dicom_meta(path):
        """
        v6: reads Laterality / ImageLaterality in the same header pass.
        Returns (inst, uid, snum, rows, cols, laterality_float).
        laterality_float: 0.0=L, 1.0=R, 0.5=unknown.
        """
        try:
            dcm  = pydicom.dcmread(str(path), stop_before_pixels=True)
            inst = int(getattr(dcm, 'InstanceNumber',    999999))
            uid  = str(getattr(dcm, 'SeriesInstanceUID', ''))
            snum = int(getattr(dcm, 'SeriesNumber',      0))
            rows = int(getattr(dcm, 'Rows',              512))
            cols = int(getattr(dcm, 'Columns',           512))
            # Try Laterality (0020,0060) first, then ImageLaterality (0020,0062)
            lat_raw = (getattr(dcm, 'Laterality', None) or
                       getattr(dcm, 'ImageLaterality', None))
            lat_map = Duke3DBBoxDataset._LAT_MAP
            lat = lat_map.get(str(lat_raw).strip().upper(), 0.5)
            return inst, uid, snum, rows, cols, lat
        except Exception:
            return 999999, '', 0, 512, 512, 0.5

    @staticmethod
    def _sort_by_instance(paths):
        """
        Filter to ONE series (prefer 601), sort by InstanceNumber.
        Returns: (paths, instance_numbers, img_rows, img_cols, laterality_float)
        v6: also returns the laterality of the chosen series.
        """
        series = defaultdict(list)  # (snum,uid) → [(inst, path, rows, cols, lat)]
        for p in paths:
            inst, uid, snum, rows, cols, lat = Duke3DBBoxDataset._read_dicom_meta(p)
            series[(snum, uid)].append((inst, p, rows, cols, lat))
        if not series:
            return [], [], 512, 512, 0.5
        best_key = None
        for key in series:
            if key[0] == 601 and len(series[key]) > 5:
                best_key = key
                break
        if best_key is None:
            best_key = max(series.keys(), key=lambda k: (len(series[k]), k[0]))
        best    = sorted(series[best_key], key=lambda x: x[0])
        paths_  = [p   for _, p,   _, _, _ in best]
        insts_  = [i   for i, _,   _, _, _ in best]
        rows_   = best[0][2]
        cols_   = best[0][3]
        # Use the majority laterality across slices (robust to occasional missing tags)
        lats    = [lat for _, _, _, _, lat in best]
        lats_known = [l for l in lats if l != 0.5]
        lat_out = round(sum(lats_known) / len(lats_known)) if lats_known else 0.5
        return paths_, insts_, rows_, cols_, float(lat_out)

    def _load_dataset(self):
        patient_folders = sorted([d for d in self.patients_dir.iterdir()
                           if d.is_dir() and not d.name.startswith('.')])
        raw_clinical, raw_imaging = [], []
        valid_samples = 0
        lat_counts = {'L': 0, 'R': 0, 'unknown': 0}
        for patient_dir in patient_folders:
            data_file = patient_dir / "patient_data.json"
            if not data_file.exists():
                continue
            try:
                with open(data_file) as f:
                    metadata = json.load(f)
                mri_dir = patient_dir / "MRI_DICOM_sample"
                if not mri_dir.exists():
                    continue
                all_dcm = list(mri_dir.glob("**/*.dcm"))
                if not all_dcm:
                    continue
                # v6: unpack 5 values (laterality added)
                sorted_dcm, instance_nums, img_rows, img_cols, laterality = \
                    self._sort_by_instance(all_dcm)
                annotations = metadata.get("annotations", [])
                if not annotations:
                    continue
                bbox_3d = annotations[0].get("bounding_box_3d", {})
                if not isinstance(bbox_3d, dict):
                    continue
                if bbox_3d.get("end_column", 0) <= bbox_3d.get("start_column", 0):
                    continue
                if bbox_3d.get("end_row", 0) <= bbox_3d.get("start_row", 0):
                    continue
                # Track laterality distribution for the sanity check
                if laterality == 0.0:
                    lat_counts['L'] += 1
                elif laterality == 1.0:
                    lat_counts['R'] += 1
                else:
                    lat_counts['unknown'] += 1
                self.samples.append(dict(
                    patient_id=patient_dir.name,
                    dicom_files=sorted_dcm,
                    instance_nums=instance_nums,
                    img_rows=img_rows,
                    img_cols=img_cols,
                    laterality=laterality,          # v6: stored per patient
                    clinical_raw=self._extract_clinical(metadata, laterality),
                    imaging_raw=self._extract_imaging(metadata),
                    bbox_3d=bbox_3d,
                ))
                valid_samples += 1
                raw_clinical.append(self.samples[-1]['clinical_raw'])
                raw_imaging.append(self.samples[-1]['imaging_raw'])
            except Exception as e:
                print(f"  SKIP {patient_dir.name}: {type(e).__name__}: {e}")
        if self.clinical_mean is None and raw_clinical:
            arr = np.array(raw_clinical, dtype=np.float32)
            self.clinical_median = np.nanmedian(arr, axis=0)
            self.clinical_mean   = np.nanmean(arr, axis=0)
            self.clinical_std    = np.nanstd(arr, axis=0) + 1e-8
        if self.imaging_mean is None and raw_imaging:
            arr = np.array(raw_imaging, dtype=np.float32)
            self.imaging_median = np.nanmedian(arr, axis=0)
            self.imaging_mean   = np.nanmean(arr, axis=0)
            self.imaging_std    = np.nanstd(arr, axis=0) + 1e-8
        print(f"Dataset: {valid_samples} patients  "
              f"clinical_dim={self.clinical_dim}  imaging_dim={self.imaging_dim}")
        print(f"Laterality — L:{lat_counts['L']}  R:{lat_counts['R']}  "
              f"unknown:{lat_counts['unknown']}")

    def _extract_clinical(self, metadata, laterality=0.5):
        """
        v6: appends laterality as the 15th (last) feature.
        laterality: 0.0=L, 1.0=R, 0.5=unknown (not normalised — it is already in [0,1]).
        """
        clin  = metadata.get("demographic_clinical", {})
        feats = []
        for key in self.CLINICAL_KEYS:
            val = clin.get(key, np.nan)
            try:
                v = float(val)
                feats.append(v if not np.isnan(v) else np.nan)
            except (TypeError, ValueError):
                feats.append(np.nan)
        grade = clin.get("Tumor Grade", 3)
        try:
            grade = int(float(grade))
        except (TypeError, ValueError):
            grade = 3
        feats.append(((grade if grade in (1, 2, 3) else 3) - 1) / 2.0)
        for key in ("ER", "PR", "HER2"):
            try:
                feats.append(float(int(clin.get(key, 0))))
            except (TypeError, ValueError):
                feats.append(0.0)
        # v6: laterality — appended last so weight surgery only touches col 27
        feats.append(float(laterality))
        return np.array(feats, dtype=np.float32)

    def _extract_imaging(self, metadata):
        img   = metadata.get("imaging_features", {})
        feats = []
        for key in self.IMAGING_KEYS:
            val = img.get(key, np.nan)
            try:
                v = float(val)
                feats.append(v if not np.isnan(v) else np.nan)
            except (TypeError, ValueError):
                feats.append(np.nan)
        return np.array(feats, dtype=np.float32)

    def _impute_normalise(self, arr, median, mean, std):
        result   = arr.copy()
        nan_mask = np.isnan(result)
        if nan_mask.any() and median is not None:
            result[nan_mask] = median[nan_mask]
        result = np.nan_to_num(result, nan=0.0)
        return np.clip((result - mean) / std, -5.0, 5.0)

    def _load_dicom_pixels(self, path):
        try:
            dcm = pydicom.dcmread(str(path))
            img = dcm.pixel_array
            while len(img.shape) > 2 and img.shape[0] == 1:
                img = img.squeeze(0)
            if len(img.shape) == 3:
                img = img[img.shape[0] // 2] if img.shape[0] < img.shape[-1] else img[:, :, 0]
            return img.astype(np.float32) if len(img.shape) == 2 else None
        except Exception:
            return None

    def _build_volume(self, dicom_files,
                      flip_x=False, flip_y=False, flip_z=False,
                      angle_deg=0.0):
        n = len(dicom_files)
        if n == 0:
            return torch.zeros(1, self.depth, self.image_size, self.image_size)
        indices = np.linspace(0, n - 1, self.depth, dtype=int)
        slices  = []
        for idx in indices:
            img = self._load_dicom_pixels(dicom_files[idx])
            if img is None:
                img = np.zeros((self.image_size, self.image_size), dtype=np.float32)
            else:
                img = np.array(
                    Image.fromarray(img).resize(
                        (self.image_size, self.image_size), Image.BILINEAR),
                    dtype=np.float32)
            slices.append(img)
        vol = np.stack(slices, axis=0)
        p1, p99 = np.percentile(vol, 1), np.percentile(vol, 99)
        if p99 > p1:
            vol = np.clip(vol, p1, p99)
        vol = (vol - vol.mean()) / (vol.std() + 1e-6)
        if flip_x:
            vol = vol[:, :, ::-1].copy()
        if flip_y:
            vol = vol[:, ::-1, :].copy()
        if flip_z:
            vol = vol[::-1, :, :].copy()
        if abs(angle_deg) > 0.1:
            S = self.image_size
            M = cv2.getRotationMatrix2D((S / 2.0, S / 2.0), angle_deg, 1.0)
            vol = np.stack(
                [cv2.warpAffine(vol[d], M, (S, S), flags=cv2.INTER_LINEAR,
                                borderMode=cv2.BORDER_REFLECT_101)
                 for d in range(self.depth)], axis=0)
        if self.augment:
            if np.random.rand() < 0.5:
                vol = vol * np.random.uniform(0.85, 1.15)
            if np.random.rand() < 0.3:
                vol = vol + np.random.normal(0, 0.02, vol.shape).astype(np.float32)
        return torch.tensor(vol, dtype=torch.float32).unsqueeze(0)

    @staticmethod
    def _rotate_bbox_xy(bbox, angle_deg):
        if abs(angle_deg) < 0.1:
            return bbox
        a = math.radians(angle_deg)
        cos_a, sin_a = math.cos(a), math.sin(a)
        x1, y1, z1, x2, y2, z2 = bbox.tolist()
        corners = [(x1, y1), (x2, y1), (x1, y2), (x2, y2)]
        rot = [(cos_a*(x-0.5)+sin_a*(y-0.5)+0.5,
               -sin_a*(x-0.5)+cos_a*(y-0.5)+0.5) for x, y in corners]
        return torch.tensor([
            max(0.0, min(c[0] for c in rot)),
            max(0.0, min(c[1] for c in rot)),
            z1,
            min(1.0, max(c[0] for c in rot)),
            min(1.0, max(c[1] for c in rot)),
            z2,
        ], dtype=torch.float32)

    def _create_bbox_target(self, bbox_3d, instance_nums, img_cols=512, img_rows=512):
        try:
            x_min    = float(bbox_3d.get("start_column", 0))
            x_max    = float(bbox_3d.get("end_column",   0))
            y_min    = float(bbox_3d.get("start_row",    0))
            y_max    = float(bbox_3d.get("end_row",      0))
            z_min_gt = bbox_3d.get("start_slice") or bbox_3d.get("start_image", 0)
            z_max_gt = bbox_3d.get("end_slice")   or bbox_3d.get("end_image",   0)
            z_min_gt, z_max_gt = float(z_min_gt), float(z_max_gt)
            if x_max <= x_min or y_max <= y_min or z_max_gt <= z_min_gt:
                return torch.zeros(6, dtype=torch.float32)
            x_min_n, x_max_n = x_min / img_cols, x_max / img_cols
            y_min_n, y_max_n = y_min / img_rows, y_max / img_rows
            valid_inst = [i for i in instance_nums if 0 <= i < 999999]
            if not valid_inst:
                n     = len(instance_nums)
                s_idx = np.linspace(0, n - 1, self.depth)
                zi    = float(np.clip(np.searchsorted(s_idx, z_min_gt), 0, self.depth-1))
                za    = float(np.clip(np.searchsorted(s_idx, z_max_gt, side='right')-1, 0, self.depth-1))
            else:
                inst_arr = np.array(valid_inst)
                idx_min  = float(np.searchsorted(inst_arr, z_min_gt + 1, side='left'))
                idx_max  = float(np.searchsorted(inst_arr, z_max_gt + 1, side='right') - 1)
                n        = len(inst_arr)
                zi = float(np.clip(idx_min*(self.depth-1)/max(n-1,1), 0, self.depth-1))
                za = float(np.clip(idx_max*(self.depth-1)/max(n-1,1), 0, self.depth-1))
            if za <= zi:
                za = min(zi + 1.0, self.depth - 1)
                zi = max(0.0, za - 1.0)
            return torch.clamp(torch.tensor([
                x_min_n, y_min_n, zi / (self.depth - 1),
                x_max_n, y_max_n, za / (self.depth - 1),
            ], dtype=torch.float32), 0.0, 1.0)
        except Exception:
            return torch.zeros(6, dtype=torch.float32)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        flip_x    = self.augment and (np.random.rand() < 0.5)
        flip_y    = self.augment and (np.random.rand() < 0.5)
        flip_z    = self.augment and (np.random.rand() < 0.5)
        angle_deg = float(np.random.uniform(-self.rot_max_deg, self.rot_max_deg)) \
                    if self.augment else 0.0
        volume = self._build_volume(
            s["dicom_files"],
            flip_x=flip_x, flip_y=flip_y, flip_z=flip_z,
            angle_deg=angle_deg)
        bbox = self._create_bbox_target(
            s["bbox_3d"], s["instance_nums"],
            img_cols=s.get("img_cols", 512),
            img_rows=s.get("img_rows", 512))
        if flip_x:
            bbox[0], bbox[3] = 1.0-bbox[3].clone(), 1.0-bbox[0].clone()
        if flip_y:
            bbox[1], bbox[4] = 1.0-bbox[4].clone(), 1.0-bbox[1].clone()
        if flip_z:
            bbox[2], bbox[5] = 1.0-bbox[5].clone(), 1.0-bbox[2].clone()
        bbox = self._rotate_bbox_xy(bbox, angle_deg)
        bbox = torch.clamp(bbox, 0.0, 1.0)
        has_bbox = torch.tensor(1.0 if bbox.sum() > 0 else 0.0, dtype=torch.float32)
        clinical = torch.tensor(
            self._impute_normalise(s["clinical_raw"], self.clinical_median,
                                   self.clinical_mean, self.clinical_std),
            dtype=torch.float32) if self.clinical_mean is not None \
            else torch.zeros(self.clinical_dim)
        imaging = torch.tensor(
            self._impute_normalise(s["imaging_raw"], self.imaging_median,
                                   self.imaging_mean, self.imaging_std),
            dtype=torch.float32) if self.imaging_mean is not None \
            else torch.zeros(self.imaging_dim)
        return {"volume": volume, "clinical": clinical, "imaging": imaging,
                "bbox_3d": bbox, "has_bbox": has_bbox, "patient_id": s["patient_id"]}

## 3. Model 

In [5]:
class SliceEncoder(nn.Module):
    """Pretrained ResNet18 applied per-slice. [B,1,D,H,W] → [B,D,512]."""
    def __init__(self, pretrained=True):
        super().__init__()
        resnet   = tv_models.resnet18(weights="IMAGENET1K_V1" if pretrained else None)
        orig_w   = resnet.conv1.weight.data
        new_conv = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        new_conv.weight.data = orig_w.mean(dim=1, keepdim=True)
        resnet.conv1 = new_conv
        self.backbone    = nn.Sequential(
            resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool,
            resnet.layer1, resnet.layer2, resnet.layer3, resnet.layer4)
        self.gap         = nn.AdaptiveAvgPool2d((1, 1))
        self.feature_dim = 512

    def forward(self, volume):
        B, C, D, H, W = volume.shape
        x = volume.permute(0, 2, 1, 3, 4).contiguous().view(B * D, C, H, W)
        return self.gap(self.backbone(x)).flatten(1).view(B, D, self.feature_dim)


class SliceAttention(nn.Module):
    """Fix 3: positional encoding + attention pooling. [B,D,F] → [B,F], [B,D]."""
    def __init__(self, feat_dim=512, hidden=64, max_depth=128):
        super().__init__()
        self.pos_embed = nn.Embedding(max_depth, feat_dim)
        nn.init.normal_(self.pos_embed.weight, std=0.02)
        self.attn = nn.Sequential(
            nn.Linear(feat_dim, hidden), nn.Tanh(), nn.Linear(hidden, 1))

    def add_positional_encoding(self, slice_feats):
        positions = torch.arange(slice_feats.shape[1], device=slice_feats.device)
        return slice_feats + self.pos_embed(positions).unsqueeze(0)

    def forward(self, pos_slice_feats):
        weights = torch.softmax(self.attn(pos_slice_feats).squeeze(-1), dim=1)
        pooled  = (weights.unsqueeze(-1) * pos_slice_feats).sum(dim=1)
        return pooled, weights


class CrossAttention(nn.Module):
    """Tabular query attends to slice features. [B,Q],[B,D,K] → [B,Q]."""
    def __init__(self, query_dim, key_dim):
        super().__init__()
        self.proj_q = nn.Linear(query_dim, key_dim)
        self.proj_v = nn.Linear(key_dim, query_dim)
        self.scale  = key_dim ** -0.5
    def forward(self, query, context):
        q = self.proj_q(query).unsqueeze(1)
        w = torch.softmax((q @ context.transpose(1, 2)) * self.scale, dim=-1)
        return query + self.proj_v((w @ context).squeeze(1))


class BBox2DSliceAttentionModel(nn.Module):
    """
    v5/v6 architecture. clinical_dim is now 15 (was 14) after adding laterality.
    All fixes (1,2,3) + v5 regularisation unchanged.
    """
    def __init__(self, clinical_dim, imaging_dim, pretrained=True, max_depth=128):
        super().__init__()
        self.slice_encoder   = SliceEncoder(pretrained=pretrained)
        self.slice_attention = SliceAttention(512, 64, max_depth=max_depth)
        self.pool_drop       = nn.Dropout(0.10)
        tab_dim = clinical_dim + imaging_dim
        self.tabular_encoder = nn.Sequential(
            nn.Linear(tab_dim, 256), nn.ReLU(True), nn.Dropout(0.25),
            nn.Linear(256, 128),     nn.ReLU(True), nn.Dropout(0.15))
        self.cross_attention  = CrossAttention(query_dim=128, key_dim=512)
        self.fusion = nn.Sequential(
            nn.Linear(640, 256), nn.ReLU(True), nn.Dropout(0.25),
            nn.Linear(256, 128), nn.ReLU(True))
        self.bbox_head = nn.Linear(128, 6)

    def get_optimizer_param_groups(self, backbone_lr=2e-4, head_lr=3e-4, wd=3e-3):
        backbone_params = list(self.slice_encoder.backbone.parameters())
        bp_ids = {id(p) for p in backbone_params}
        other  = [p for p in self.parameters() if id(p) not in bp_ids]
        return [{"params": backbone_params, "lr": backbone_lr, "weight_decay": wd},
                {"params": other,           "lr": head_lr,     "weight_decay": wd}]

    @staticmethod
    def decode_bbox(raw):
        sig  = torch.sigmoid(raw)
        mins = torch.min(sig[:, :3], sig[:, 3:])
        maxs = torch.max(sig[:, :3], sig[:, 3:])
        return torch.cat([mins, maxs], dim=1)

    def forward(self, volume, clinical, imaging):
        sf       = self.slice_encoder(volume)
        sf_pos   = self.slice_attention.add_positional_encoding(sf)
        img_feat, attn_w = self.slice_attention(sf_pos)
        img_feat = self.pool_drop(img_feat)
        tab      = self.tabular_encoder(torch.cat([clinical, imaging], dim=1))
        tab      = self.cross_attention(tab, sf_pos)
        fused    = self.fusion(torch.cat([img_feat, tab], dim=1))
        return {"bbox_3d": self.decode_bbox(self.bbox_head(fused)),
                "slice_attn": attn_w}

print("Model defined.")

Model v6 defined.


## 4. Loss functions 

In [6]:
def bbox_iou_3d(pred, target):
    px1,py1,pz1,px2,py2,pz2 = pred[:,0],pred[:,1],pred[:,2],pred[:,3],pred[:,4],pred[:,5]
    tx1,ty1,tz1,tx2,ty2,tz2 = target[:,0],target[:,1],target[:,2],target[:,3],target[:,4],target[:,5]
    iw   = torch.clamp(torch.min(px2,tx2)-torch.max(px1,tx1), min=0)
    ih   = torch.clamp(torch.min(py2,ty2)-torch.max(py1,ty1), min=0)
    id_  = torch.clamp(torch.min(pz2,tz2)-torch.max(pz1,tz1), min=0)
    inter = iw*ih*id_
    pv = torch.clamp(px2-px1,min=0)*torch.clamp(py2-py1,min=0)*torch.clamp(pz2-pz1,min=0)
    tv = torch.clamp(tx2-tx1,min=0)*torch.clamp(ty2-ty1,min=0)*torch.clamp(tz2-tz1,min=0)
    return inter/(pv+tv-inter+1e-6)

def bbox_giou_3d(pred, target):
    px1,py1,pz1,px2,py2,pz2 = pred[:,0],pred[:,1],pred[:,2],pred[:,3],pred[:,4],pred[:,5]
    tx1,ty1,tz1,tx2,ty2,tz2 = target[:,0],target[:,1],target[:,2],target[:,3],target[:,4],target[:,5]
    iw  = torch.clamp(torch.min(px2,tx2)-torch.max(px1,tx1), min=0)
    ih  = torch.clamp(torch.min(py2,ty2)-torch.max(py1,ty1), min=0)
    id_ = torch.clamp(torch.min(pz2,tz2)-torch.max(pz1,tz1), min=0)
    inter = iw*ih*id_
    pv    = torch.clamp(px2-px1,min=0)*torch.clamp(py2-py1,min=0)*torch.clamp(pz2-pz1,min=0)
    tv    = torch.clamp(tx2-tx1,min=0)*torch.clamp(ty2-ty1,min=0)*torch.clamp(tz2-tz1,min=0)
    union = pv+tv-inter+1e-6
    iou   = inter/union
    ew = torch.clamp(torch.max(px2,tx2)-torch.min(px1,tx1), min=0)
    eh = torch.clamp(torch.max(py2,ty2)-torch.min(py1,ty1), min=0)
    ed = torch.clamp(torch.max(pz2,tz2)-torch.min(pz1,tz1), min=0)
    return iou-(ew*eh*ed+1e-6-union)/(ew*eh*ed+1e-6)

def bbox_diou_3d(pred, target):
    iou       = bbox_iou_3d(pred, target)
    pred_c    = (pred[:,:3]+pred[:,3:])/2.0
    gt_c      = (target[:,:3]+target[:,3:])/2.0
    dist2     = ((pred_c-gt_c)**2).sum(dim=1)
    enc_min   = torch.min(pred[:,:3], target[:,:3])
    enc_max   = torch.max(pred[:,3:], target[:,3:])
    enc_diag2 = ((enc_max-enc_min)**2).sum(dim=1).clamp(min=1e-6)
    return iou-dist2/enc_diag2

def center_distance_3d(pred, target):
    pred_c   = (pred[:,:3]+pred[:,3:])/2.0
    gt_c     = (target[:,:3]+target[:,3:])/2.0
    dist_abs = torch.norm(pred_c-gt_c, dim=1)
    gt_diag  = torch.norm(target[:,3:]-target[:,:3], dim=1).clamp(min=1e-6)
    return dist_abs, dist_abs/gt_diag

print("Loss / metric functions defined.")

Loss / metric functions defined.


## 5. Trainer 

In [7]:
class BBox3DTrainerV6:
    def __init__(self, model, device,
                 backbone_lr=2e-4, head_lr=3e-4, weight_decay=3e-3,
                 lambda_l1=0.20, lambda_iou=0.25, lambda_giou=0.10,
                 lambda_diou=0.25, lambda_center=0.20,
                 warmup_epochs=5, total_epochs=200):
        self.model         = model
        self.device        = device
        self.lambda_l1     = lambda_l1
        self.lambda_iou    = lambda_iou
        self.lambda_giou   = lambda_giou
        self.lambda_diou   = lambda_diou
        self.lambda_center = lambda_center
        self.warmup_epochs = warmup_epochs
        self.total_epochs  = total_epochs
        self.l1_loss       = nn.SmoothL1Loss(reduction="none", beta=0.1)
        if hasattr(model, "get_optimizer_param_groups"):
            pgs = model.get_optimizer_param_groups(backbone_lr, head_lr, weight_decay)
        else:
            pgs = [{"params": model.parameters(), "lr": head_lr, "weight_decay": weight_decay}]
        for pg in pgs:
            pg["initial_lr"] = pg["lr"]
        self.optimizer = optim.AdamW(pgs)
        self.scaler    = torch.cuda.amp.GradScaler(enabled=(device.type == "cuda"))
        self.history   = defaultdict(list)
        self.best_iou  = -1.0

    def _lr_scale(self, epoch):
        if epoch < self.warmup_epochs:
            return (epoch+1)/self.warmup_epochs
        p = (epoch-self.warmup_epochs)/max(1, self.total_epochs-self.warmup_epochs)
        return 0.5*(1.0+math.cos(math.pi*p))

    def _update_lr(self, epoch):
        s = self._lr_scale(epoch)
        for pg in self.optimizer.param_groups:
            pg["lr"] = pg["initial_lr"]*s

    def _compute_loss(self, pred, gt, has_bbox):
        mask  = has_bbox.float()
        denom = mask.sum().clamp(min=1.0)
        l1    = (self.l1_loss(pred, gt).mean(dim=1)*mask).sum()/denom
        iou   = bbox_iou_3d(pred, gt)
        giou  = bbox_giou_3d(pred, gt)
        diou  = bbox_diou_3d(pred, gt)
        dist_abs, dist_rel = center_distance_3d(pred, gt)
        iou_l  = ((1.0-iou) *mask).sum()/denom
        giou_l = ((1.0-giou)*mask).sum()/denom
        diou_l = ((1.0-diou)*mask).sum()/denom
        cd_l   = (dist_abs   *mask).sum()/denom
        total  = (self.lambda_l1*l1 + self.lambda_iou*iou_l +
                  self.lambda_giou*giou_l + self.lambda_diou*diou_l +
                  self.lambda_center*cd_l)
        pred_z_c = (pred[:,2]+pred[:,5])/2.0
        gt_z_c   = (gt[:,2]+gt[:,5])/2.0
        z_dist   = (torch.abs(pred_z_c-gt_z_c)*mask).sum()/denom
        return total, iou.detach(), dist_abs.detach(), dist_rel.detach(), z_dist.detach()

    def _run_epoch(self, loader, train=True):
        self.model.train(train)
        total_loss = 0.0
        all_iou = all_dist = all_drel = all_zdist = []
        ctx = torch.enable_grad() if train else torch.no_grad()
        with ctx:
            for batch in loader:
                vol  = batch["volume"].to(self.device,   non_blocking=True)
                clin = batch["clinical"].to(self.device, non_blocking=True)
                img  = batch["imaging"].to(self.device,  non_blocking=True)
                gt   = batch["bbox_3d"].to(self.device,  non_blocking=True)
                hb   = batch["has_bbox"].to(self.device, non_blocking=True)
                if train:
                    self.optimizer.zero_grad(set_to_none=True)
                with torch.cuda.amp.autocast(enabled=(self.device.type=="cuda")):
                    out  = self.model(vol, clin, img)
                    pred = out["bbox_3d"]
                    loss, iou, dist_abs, dist_rel, z_dist = \
                        self._compute_loss(pred, gt, hb)
                if train:
                    self.scaler.scale(loss).backward()
                    self.scaler.unscale_(self.optimizer)
                    torch.nn.utils.clip_grad_norm_(self.model.parameters(), 1.0)
                    self.scaler.step(self.optimizer)
                    self.scaler.update()
                total_loss += loss.item()
                valid = hb > 0
                if valid.any():
                    all_iou   = all_iou   + iou[valid].cpu().tolist()
                    all_dist  = all_dist  + dist_abs[valid].cpu().tolist()
                    all_drel  = all_drel  + dist_rel[valid].cpu().tolist()
                    all_zdist = all_zdist + z_dist.unsqueeze(0).cpu().tolist()
        return (
            total_loss/len(loader),
            float(np.mean(all_iou))   if all_iou   else 0.0,
            float(np.mean(all_dist))  if all_dist  else 1.0,
            float(np.mean(all_drel))  if all_drel  else 1.0,
            float(np.mean(all_zdist)) if all_zdist else 1.0,
        )

    def train(self, train_loader, val_loader, num_epochs=200, patience=30,
              save_path="best_3d_bbox_model_v6.pth"):
        print("\n" + "="*70)
        print("TRAINING v6: laterality feature + 200 epochs warm-start")
        print("="*70)
        no_improve = 0
        for epoch in range(num_epochs):
            self._update_lr(epoch)
            tr = self._run_epoch(train_loader, train=True)
            va = self._run_epoch(val_loader,   train=False)
            for key, val in zip(
                ["train_loss","train_iou","train_center_dist",
                 "train_center_dist_rel","train_z_dist"], tr):
                self.history[key].append(val)
            for key, val in zip(
                ["val_loss","val_iou","val_center_dist",
                 "val_center_dist_rel","val_z_dist"], va):
                self.history[key].append(val)
            lr  = self.optimizer.param_groups[-1]["lr"]
            gap = tr[1]/max(va[1], 1e-6)
            print(f"\nEpoch {epoch+1}/{num_epochs}  LR={lr:.2e}  Gap={gap:.1f}×")
            print(f"  Train  loss:{tr[0]:.4f}  IoU:{tr[1]:.4f}  "
                  f"CD:{tr[2]:.4f}  ZDist:{tr[4]:.4f}")
            print(f"  Val    loss:{va[0]:.4f}  IoU:{va[1]:.4f}  "
                  f"CD:{va[2]:.4f}  ZDist:{va[4]:.4f}")
            if va[1] > self.best_iou:
                self.best_iou = va[1]
                no_improve    = 0
                torch.save({"epoch": epoch,
                            "model_state_dict": self.model.state_dict(),
                            "best_iou": self.best_iou}, save_path)
                print("  ✓ Best model saved!")
            else:
                no_improve += 1
                print(f"  Patience: {no_improve}/{patience}")
            if no_improve >= patience:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break
        ckpt = torch.load(save_path, map_location=self.device)
        self.model.load_state_dict(ckpt["model_state_dict"])
        print(f"\nBest Val IoU: {self.best_iou:.4f}")

print("Trainer defined.")

Trainer v6 defined.


## 6. Visualization

In [8]:
def plot_training_curves(history, save_path="results/3d_bbox_training_curves.png"):
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(2, 3, figsize=(18, 8))
    panels = [
        ("train_loss",            "val_loss",            "Loss"),
        ("train_iou",             "val_iou",             "3D IoU"),
        ("train_center_dist",     "val_center_dist",     "Center Distance (abs)"),
        ("train_center_dist_rel", "val_center_dist_rel", "Center Distance (relative)"),
        ("train_z_dist",          "val_z_dist",          "Z-axis Distance (Fix 3)"),
    ]
    for (tk, vk, title), ax in zip(panels, axes.flatten()):
        ax.plot(history[tk], label="Train", linewidth=2)
        ax.plot(history[vk], label="Val",   linewidth=2, linestyle="--")
        if "iou" in tk and len(history[tk]) > 0:
            gap = history[tk][-1]/max(history[vk][-1], 1e-6)
            ax.set_title(f"{title}  (gap {gap:.1f}×)", fontsize=10, fontweight="bold")
        else:
            ax.set_title(title, fontsize=10, fontweight="bold")
        ax.set_xlabel("Epoch"); ax.legend(); ax.grid(True, alpha=0.3)
    axes.flatten()[-1].axis('off')
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")


@torch.no_grad()
def evaluate_and_visualize(model, loader, device, num_samples=8,
                            save_path="results/3d_bbox_predictions.png"):
    Path(save_path).parent.mkdir(parents=True, exist_ok=True)
    model.eval()
    rows = []; viz_data = []
    for batch in loader:
        vol  = batch["volume"].to(device)
        clin = batch["clinical"].to(device)
        img  = batch["imaging"].to(device)
        gt   = batch["bbox_3d"]
        ids  = batch["patient_id"]
        out  = model(vol, clin, img)
        pred = out["bbox_3d"].cpu()
        iou  = bbox_iou_3d(pred, gt)
        da, dr = center_distance_3d(pred, gt)
        for i in range(len(ids)):
            rows.append(dict(patient=ids[i], iou=iou[i].item(),
                             center_dist=da[i].item(), center_dist_rel=dr[i].item(),
                             pred=pred[i].numpy().round(3).tolist(),
                             gt=gt[i].numpy().round(3).tolist()))
            if len(viz_data) < num_samples:
                viz_data.append(dict(pid=ids[i], vol=batch["volume"][i,0].numpy(),
                                     gt=gt[i].numpy(), pred=pred[i].numpy(),
                                     iou=iou[i].item(), cd=da[i].item()))
    print(f"\n{'Patient':<22} {'IoU':>6} {'CenterDist':>11} {'RelDist':>9}  Pred → GT")
    print("-"*100)
    for r in rows:
        print(f"{r['patient']:<22} {r['iou']:6.3f} {r['center_dist']:11.4f} "
              f"{r['center_dist_rel']:9.4f}  {r['pred']} → {r['gt']}")
    mi  = np.mean([r["iou"]             for r in rows])
    md_ = np.mean([r["center_dist"]     for r in rows])
    mr  = np.mean([r["center_dist_rel"] for r in rows])
    print("-"*100)
    print(f"{'MEAN':<22} {mi:6.3f} {md_:11.4f} {mr:9.4f}")
    n = len(viz_data)
    fig, axes = plt.subplots(n, 3, figsize=(12, 4*n))
    if n == 1: axes = axes[np.newaxis,:]
    view_names = ["Axial (XY)", "Coronal (XZ)", "Sagittal (YZ)"]
    for ri, d in enumerate(viz_data):
        vol = d["vol"]; gt = d["gt"]; pred = d["pred"]
        D,H,W = vol.shape
        gx1,gy1,gz1,gx2,gy2,gz2 = gt[0]*W,gt[1]*H,gt[2]*D,gt[3]*W,gt[4]*H,gt[5]*D
        px1,py1,pz1,px2,py2,pz2 = pred[0]*W,pred[1]*H,pred[2]*D,pred[3]*W,pred[4]*H,pred[5]*D
        z_c = int(np.clip((gz1+gz2)/2, 0, D-1))
        views = [
            (vol[z_c,:,:],  (gx1,gy1,gx2-gx1,gy2-gy1), (px1,py1,px2-px1,py2-py1)),
            (vol[:,H//2,:], (gx1,gz1,gx2-gx1,gz2-gz1), (px1,pz1,px2-px1,pz2-pz1)),
            (vol[:,:,W//2], (gz1,gy1,gz2-gz1,gy2-gy1), (pz1,py1,pz2-pz1,py2-py1)),
        ]
        for ci, (slc,gr,pr) in enumerate(views):
            ax = axes[ri,ci]
            s  = (slc-slc.min())/(slc.max()-slc.min()+1e-8)
            ax.imshow(s, cmap="gray", origin="upper")
            for (x,y,w,h),col,ls,lbl in [(gr,"lime","-","GT"),(pr,"red","--","Pred")]:
                if w>0 and h>0:
                    ax.add_patch(patches.Rectangle((x,y),w,h,fill=False,
                                 edgecolor=col,linewidth=2,linestyle=ls,label=lbl))
            title = (f"{d['pid']}\n{view_names[ci]}  IoU={d['iou']:.3f} CD={d['cd']:.3f}"
                     if ci==0 else view_names[ci])
            ax.set_title(title, fontsize=8); ax.axis("off")
            if ci==0: ax.legend(loc="upper right", fontsize=6)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches="tight")
    plt.close()
    print(f"Saved: {save_path}")
    return rows

print("Visualization defined.")

Visualization defined.


## 7. Run

In [9]:
EXPORT_DIR   = "exported_patients"
DEPTH        = 64
IMAGE_SIZE   = 192
BATCH_SIZE   = 4       # reduce to 2 if OOM
NUM_EPOCHS   = 200     # v6: extended from 120
PATIENCE     = 30      # v6: extended from 20
PRETRAINED   = True
NUM_WORKERS  = 0       # 0 avoids /dev/shm exhaustion with large tensors
MAX_DEPTH    = 128
ROT_MAX_DEG  = 10.0
V5_CKPT      = "best_3d_bbox_model_v5.pth"  # warm-start source

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}  DEPTH={DEPTH}  IMAGE_SIZE={IMAGE_SIZE}  Batch={BATCH_SIZE}")
if torch.cuda.is_available():
    vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"VRAM {vram_gb:.0f} GB — {'OK for batch 4' if vram_gb >= 20 else 'consider batch 2'}")

Device: cuda  DEPTH=64  IMAGE_SIZE=192  Batch=4
VRAM 25 GB — OK for batch 4


In [10]:
print("Loading train dataset...")
train_ds = Duke3DBBoxDataset(
    Path(EXPORT_DIR) / "train",
    depth=DEPTH, image_size=IMAGE_SIZE,
    augment=True, rot_max_deg=ROT_MAX_DEG)

print("\nLoading val dataset...")
val_ds = Duke3DBBoxDataset(
    Path(EXPORT_DIR) / "test",
    depth=DEPTH, image_size=IMAGE_SIZE,
    augment=False, rot_max_deg=0.0,
    clinical_median=train_ds.clinical_median,
    clinical_mean=train_ds.clinical_mean,
    clinical_std=train_ds.clinical_std,
    imaging_median=train_ds.imaging_median,
    imaging_mean=train_ds.imaging_mean,
    imaging_std=train_ds.imaging_std)
val_ds.clinical_dim = train_ds.clinical_dim
val_ds.imaging_dim  = train_ds.imaging_dim

print(f"\nTrain: {len(train_ds)}  |  Val: {len(val_ds)}")
print(f"clinical_dim: {train_ds.clinical_dim}") #should be 15

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=False)

Loading train dataset (v6 — laterality feature added)...
Dataset: 727 patients  clinical_dim=15  imaging_dim=13
Laterality — L:0  R:0  unknown:727

Loading val dataset...
Dataset: 175 patients  clinical_dim=15  imaging_dim=13
Laterality — L:0  R:0  unknown:175

Train: 727  |  Val: 175
clinical_dim: 15  ← should be 15 (was 14)


In [11]:
# Print the laterality values for the first 10 val patients L/R is being read correctly
np.random.seed(42)
print("Laterality for first 10 val patients:")
for s in val_ds.samples[:10]:
    lat_raw = s['laterality']
    lat_str = 'L' if lat_raw == 0.0 else ('R' if lat_raw == 1.0 else 'unknown')
    # The feature is the last element of clinical_raw
    print(f"  {s['patient_id']:<20}  laterality={lat_str} ({lat_raw})")

# Check that clinical_raw has 15 elements
s0 = val_ds.samples[0]
print(f"\nclinical_raw length: {len(s0['clinical_raw'])}  ← should be 15")
print(f"Last feature (laterality): {s0['clinical_raw'][-1]}")

Laterality for first 10 val patients:
  Patient_001           laterality=unknown (0.5)
  Patient_005           laterality=unknown (0.5)
  Patient_006           laterality=unknown (0.5)
  Patient_014           laterality=unknown (0.5)
  Patient_016           laterality=unknown (0.5)
  Patient_021           laterality=unknown (0.5)
  Patient_026           laterality=unknown (0.5)
  Patient_029           laterality=unknown (0.5)
  Patient_033           laterality=unknown (0.5)
  Patient_036           laterality=unknown (0.5)

clinical_raw length: 15  ← should be 15
Last feature (laterality): 0.5


In [12]:
# Build model and perform weight surgery from checkpoint
save_path = "best_3d_bbox_model_v6.pth"

model = BBox2DSliceAttentionModel(
    clinical_dim=train_ds.clinical_dim,   # 15
    imaging_dim=train_ds.imaging_dim,     # 13
    pretrained=PRETRAINED,
    max_depth=MAX_DEPTH).to(device)

# Weight surgery: warm-start from previous model, adapt tabular_encoder.0 
if Path(V5_CKPT).exists():
    print(f"Loading v5 checkpoint: {V5_CKPT}")
    ckpt_v5  = torch.load(V5_CKPT, map_location=device)
    v5_state = ckpt_v5['model_state_dict']
    new_state = model.state_dict()

    copied = skipped = surgery = 0
    for k, v5_w in v5_state.items():
        if k not in new_state:
            skipped += 1
            continue
        new_w = new_state[k]
        if new_w.shape == v5_w.shape:
            # All layers whose size is unchanged — copy exactly
            new_state[k] = v5_w
            copied += 1
        elif k == 'tabular_encoder.0.weight':
            # [256, 27]  →  [256, 28]
            # Copy first 27 columns; 28th column (laterality) stays zero-init
            # Zero-init is deliberate: the model starts neutral on the new feature
            # and learns its weight from data. Random-init would inject noise.
            new_state[k][:, :v5_w.shape[1]] = v5_w
            print(f"  Weight surgery: {k}  {list(v5_w.shape)} → {list(new_w.shape)}")
            surgery += 1
        else:
            print(f"  Shape mismatch (skipped): {k}  {list(v5_w.shape)} vs {list(new_w.shape)}")
            skipped += 1

    model.load_state_dict(new_state)
    print(f"Warm-start complete: {copied} copied, {surgery} surgery, {skipped} skipped")
    print(f"Starting from v5 best IoU: {ckpt_v5.get('best_iou', 'unknown'):.4f}")
else:
    print(f"WARNING: {V5_CKPT} not found — training from scratch with pretrained backbone.")

total     = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {total:,} params ({trainable:,} trainable)")

Loading v5 checkpoint: best_3d_bbox_model_v5.pth
  Weight surgery: tabular_encoder.0.weight  [256, 27] → [256, 28]
Warm-start complete: 138 copied, 1 surgery, 0 skipped
Starting from v5 best IoU: 0.2332
Model: 11,638,471 params (11,638,471 trainable)


In [ ]:
trainer = BBox3DTrainerV6(
    model=model, device=device,
    backbone_lr=2e-4, head_lr=3e-4,
    weight_decay=3e-3,
    lambda_l1=0.20, lambda_iou=0.25, lambda_giou=0.10,
    lambda_diou=0.25, lambda_center=0.20,
    warmup_epochs=5, total_epochs=NUM_EPOCHS)

trainer.train(train_loader, val_loader,
              num_epochs=NUM_EPOCHS, patience=PATIENCE,
              save_path=save_path)


TRAINING v6: laterality feature + 200 epochs warm-start

Epoch 1/200  LR=6.00e-05  Gap=1.1×
  Train  loss:0.5242  IoU:0.2063  CD:0.0726  ZDist:0.0449
  Val    loss:0.5835  IoU:0.1802  CD:0.1343  ZDist:0.0705
  ✓ Best model saved!

Epoch 2/200  LR=1.20e-04  Gap=1.2×
  Train  loss:0.5247  IoU:0.2128  CD:0.0798  ZDist:0.0481
  Val    loss:0.5783  IoU:0.1831  CD:0.1313  ZDist:0.0692
  ✓ Best model saved!

Epoch 3/200  LR=1.80e-04  Gap=1.1×
  Train  loss:0.5349  IoU:0.2049  CD:0.0875  ZDist:0.0484
  Val    loss:0.5726  IoU:0.1913  CD:0.1354  ZDist:0.0687
  ✓ Best model saved!

Epoch 4/200  LR=2.40e-04  Gap=1.1×
  Train  loss:0.5429  IoU:0.1991  CD:0.0946  ZDist:0.0505
  Val    loss:0.5897  IoU:0.1751  CD:0.1376  ZDist:0.0674
  Patience: 1/30

Epoch 5/200  LR=3.00e-04  Gap=1.1×
  Train  loss:0.5449  IoU:0.1925  CD:0.0912  ZDist:0.0519
  Val    loss:0.5852  IoU:0.1806  CD:0.1422  ZDist:0.0731
  Patience: 2/30

Epoch 6/200  LR=3.00e-04  Gap=1.0×
  Train  loss:0.5503  IoU:0.1870  CD:0.0986  ZD

In [ ]:
plot_training_curves(trainer.history)

results = evaluate_and_visualize(
    model, val_loader, device, num_samples=8,
    save_path="results/3d_bbox_predictions.png")

print(f"\nBest Val IoU : {trainer.best_iou:.4f}")
print(f"Model saved  : {save_path}")
print()
print("Laterality check: P_001, P_006, P_026, P_029 should now have IoU > 0.")
print("If they still show IoU=0, check the laterality sanity cell above —")
print("  all 0.5 (unknown) means the DICOM tag is absent in this dataset.")
print()
print("If val IoU > 0.35 and gap > 2×: raise weight_decay to 5e-3.")
print("If val IoU > 0.40: next step is ResNet34 backbone.")